# Fase 4 - Desarrollo de la Aplicación

## Aplicación Predictiva de Carga Energética

Curso: Introducción a la Inteligencia Artificial

## 1. Objetivo

Integrar el modelo de red neuronal (MLPRegressor) entrenado en la Fase 3 en una aplicación funcional que permita:
- Ingresar datos nuevos de características arquitectónicas
- Ejecutar el modelo entrenado
- Mostrar el resultado de la predicción de forma clara

## 2. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Librerías importadas correctamente')

Librerías importadas correctamente


## 3. Cargar y preparar los datos

Cargamos los datos originales para ajustar el escalador y los datos estandarizados para entrenar el modelo.

In [2]:
# Ajustar ruta base: si estamos en notebooks/, subimos un nivel
BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))

RAW_PATH = os.path.join(BASE, 'data', 'raw', 'ENB2012_data.xlsx')
CLEAN_PATH = os.path.join(BASE, 'data', 'processed', 'data_clean.csv')

df_raw = pd.read_excel(RAW_PATH)
print(f'Dataset original cargado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
df_raw.head()


Dataset original cargado: 768 filas x 10 columnas


,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


In [3]:
X_raw = df_raw[['X1','X2','X3','X4','X5','X6','X7','X8']]

scaler = StandardScaler()
scaler.fit(X_raw)

print('StandardScaler ajustado con datos originales')

StandardScaler ajustado con datos originales


In [4]:
df_clean = pd.read_csv(CLEAN_PATH)
X = df_clean[['X1','X2','X3','X4','X5','X6','X7','X8']]
y = df_clean['Y1']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba:        {X_test.shape[0]} muestras')

Entrenamiento: 614 muestras
Prueba:        154 muestras


## 4. Entrenar el modelo MLPRegressor

Utilizamos la misma arquitectura de la Fase 3: dos capas ocultas de 64 y 32 neuronas, activación ReLU, optimizador Adam.

In [5]:
modelo = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=3000,
    random_state=42
)

modelo.fit(X_train, y_train)
print('Modelo entrenado exitosamente')

Modelo entrenado exitosamente


## 5. Evaluar el modelo en test

Verificamos que el rendimiento coincida con lo obtenido en la Fase 3.

In [7]:
y_pred = modelo.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('=' * 35)
print(f'  MAE  : {mae:.4f}')
print(f'  RMSE : {rmse:.4f}')
print(f'  R²   : {r2:.4f}')
print('=' * 35)

  MAE  : 0.4505
  RMSE : 0.6039
  R²   : 0.9965


## 6. Guardar el modelo y el escalador

Persistimos ambos objetos para usarlos en la aplicación sin necesidad de reentrenar.

In [9]:
MODEL_DIR = os.path.join(BASE, 'src', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(modelo, os.path.join(MODEL_DIR, 'mlp_model.pkl'))
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))

print('Modelo guardado en src/models/mlp_model.pkl')
print('Escalador guardado en src/models/scaler.pkl')

Modelo guardado en src/models/mlp_model.pkl
Escalador guardado en src/models/scaler.pkl


---
## 7. FUNCIÓN PREDICTIVA

Función que recibe valores originales de las 8 variables arquitectónicas, los estandariza y devuelve la predicción de Heating Load (Y1).

In [10]:
FEATURE_NAMES = [
    'X1: Compacidad relativa',
    'X2: Área de superficie',
    'X3: Área de pared',
    'X4: Área de techo',
    'X5: Altura total',
    'X6: Orientación',
    'X7: Área de acristalamiento',
    'X8: Distribución del acristalamiento',
]

FEATURES = ['X1','X2','X3','X4','X5','X6','X7','X8']

def predecir_carga(valores):
    X_raw = pd.DataFrame([valores], columns=FEATURES)
    X_scaled = scaler.transform(X_raw)
    pred = modelo.predict(pd.DataFrame(X_scaled, columns=FEATURES))[0]
    return pred

def clasificar(pred):
    if pred < 22:
        return 'Baja demanda energética'
    elif pred < 31:
        return 'Demanda energética media'
    else:
        return 'Alta demanda energética'

def mostrar_prediccion(valores, pred, real=None):
    print()
    print('-' * 55)
    print('VALORES DE ENTRADA')
    print('-' * 55)
    for name, val in zip(FEATURE_NAMES, valores):
        print(f'  {name}: {val}')
    print()
    print('-' * 55)
    print('RESULTADO DE LA PREDICCIÓN')
    print('-' * 55)
    print(f'  Heating Load (Y1) estimado: {pred:.2f}')
    if real is not None:
        print(f'  Valor real (Y1):             {real:.2f}')
        print(f'  Error absoluto:              {abs(pred - real):.2f}')
    print(f'  Clasificación:               {clasificar(pred)}')
    print('-' * 55)

print('Funciones cargadas correctamente')

Funciones cargadas correctamente


---
## 8. EJEMPLOS DE PREDICCIÓN

### 8.1 Ejemplo 1: Primera fila del dataset

In [11]:
fila = df_raw.iloc[0]
valores = [float(fila[f]) for f in FEATURES]
real = float(fila['Y1'])

pred = predecir_carga(valores)
mostrar_prediccion(valores, pred, real)


-------------------------------------------------------
VALORES DE ENTRADA
-------------------------------------------------------
  X1: Compacidad relativa: 0.98
  X2: Área de superficie: 514.5
  X3: Área de pared: 294.0
  X4: Área de techo: 110.25
  X5: Altura total: 7.0
  X6: Orientación: 2.0
  X7: Área de acristalamiento: 0.0
  X8: Distribución del acristalamiento: 0.0

-------------------------------------------------------
RESULTADO DE LA PREDICCIÓN
-------------------------------------------------------
  Heating Load (Y1) estimado: 16.34
  Valor real (Y1):             15.55
  Error absoluto:              0.79
  Clasificación:               Baja demanda energética
-------------------------------------------------------


### 8.2 Ejemplo 2: Muestra aleatoria del dataset

In [12]:
fila = df_raw.sample(1, random_state=123).iloc[0]
valores = [float(fila[f]) for f in FEATURES]
real = float(fila['Y1'])

pred = predecir_carga(valores)
mostrar_prediccion(valores, pred, real)


-------------------------------------------------------
VALORES DE ENTRADA
-------------------------------------------------------
  X1: Compacidad relativa: 0.62
  X2: Área de superficie: 808.5
  X3: Área de pared: 367.5
  X4: Área de techo: 220.5
  X5: Altura total: 3.5
  X6: Orientación: 2.0
  X7: Área de acristalamiento: 0.1
  X8: Distribución del acristalamiento: 4.0

-------------------------------------------------------
RESULTADO DE LA PREDICCIÓN
-------------------------------------------------------
  Heating Load (Y1) estimado: 12.50
  Valor real (Y1):             12.85
  Error absoluto:              0.35
  Clasificación:               Baja demanda energética
-------------------------------------------------------


### 8.3 Ejemplo 3: Datos personalizados (edificio hipotético)

Modifica los valores en la siguiente celda para probar con tus propios datos.

In [13]:
valores_personalizados = [0.75, 650.0, 310.0, 180.0, 3.5, 3, 2, 3]

pred = predecir_carga(valores_personalizados)
mostrar_prediccion(valores_personalizados, pred)


-------------------------------------------------------
VALORES DE ENTRADA
-------------------------------------------------------
  X1: Compacidad relativa: 0.75
  X2: Área de superficie: 650.0
  X3: Área de pared: 310.0
  X4: Área de techo: 180.0
  X5: Altura total: 3.5
  X6: Orientación: 3
  X7: Área de acristalamiento: 2
  X8: Distribución del acristalamiento: 3

-------------------------------------------------------
RESULTADO DE LA PREDICCIÓN
-------------------------------------------------------
  Heating Load (Y1) estimado: 55.21
  Clasificación:               Alta demanda energética
-------------------------------------------------------


---
## 9. PREDICCIÓN POR LOTES (BATCH)

Evaluamos el modelo sobre todo el dataset y comparamos predicciones vs valores reales.

In [14]:
X_all_raw = df_raw[FEATURES]
X_all_scaled = scaler.transform(X_all_raw)
pred_todas = modelo.predict(pd.DataFrame(X_all_scaled, columns=FEATURES))

resultado = pd.DataFrame({
    'Real': df_raw['Y1'],
    'Predicho': pred_todas.round(2),
    'Error': abs(df_raw['Y1'] - pred_todas).round(2)
})

In [15]:
print('=== 10 predicciones con mayor error ===')
display(resultado.sort_values('Error', ascending=False).head(10))

=== 10 predicciones con mayor error ===


,Real,Predicho,Error
271,13.69,11.10,2.59
494,24.60,26.54,1.94
23,23.93,25.87,1.94
396,24.70,26.48,1.78
15,15.98,17.67,1.69
572,15.09,16.77,1.68
748,12.43,14.09,1.66
445,24.96,26.53,1.57
628,34.24,35.78,1.54
493,25.17,26.61,1.44


In [16]:
print('=== 10 predicciones con menor error ===')
display(resultado.sort_values('Error', ascending=True).head(10))

=== 10 predicciones con menor error ===


,Real,Predicho,Error
427,16.86,16.86,0.00
37,7.10,7.10,0.00
87,11.69,11.69,0.00
286,12.80,12.80,0.00
598,40.57,40.57,0.00
191,12.73,12.73,0.00
582,36.06,36.06,0.00
372,13.05,13.05,0.00
220,10.66,10.66,0.00
231,11.43,11.44,0.01


In [17]:
print('=== Resumen general del error ===')
print(f'Error promedio: {resultado["Error"].mean():.2f}')
print(f'Error máximo:  {resultado["Error"].max():.2f}')
print(f'Error mínimo:  {resultado["Error"].min():.2f}')

=== Resumen general del error ===
Error promedio: 0.36
Error máximo:  2.59
Error mínimo:  0.00


---
## 10. CONCLUSIÓN

La aplicación desarrollada integra exitosamente el modelo de red neuronal (MLPRegressor) entrenado en la Fase 3, permitiendo:

1. **Ingresar datos manualmente**: 8 variables arquitectónicas en sus escalas originales
2. **Estandarización automática**: Los valores se transforman usando el mismo StandardScaler de la Fase 1
3. **Predicción precisa**: El modelo alcanza un R² de 0.9965 en el conjunto de prueba
4. **Resultados claros**: Se muestra el valor estimado de Heating Load (Y1) junto con una clasificación del nivel de demanda energética
